# Week 7 capstone: Step 3, GPU smoke test and determinism-policy timing

**Scope of this notebook: smoke stage only.** It contains no calibration, pilot, or main-experiment cells, and the repository refuses to run those stages until the policy decision, the environment lock, and (for main runs) the frozen pre-registration exist. All experiment logic lives in the repository. This notebook only clones it and calls its scripts, so cell order cannot change what the experiment does.

**This notebook never runs `pip install` or upgrades anything.** It records the packages Colab provides and checks them against `requirements.lock.txt`.

Run the cells top to bottom, on a **T4 GPU runtime** (Runtime > Change runtime type). Then download the zip from the last cell and send it back for review. Do not start calibration until the outputs have been reviewed.

In [ ]:
# 1. Get the code. Edit REPO_URL if yours differs. Optionally pin COMMIT to an exact commit hash.
REPO_URL = "https://github.com/sweiss1367/wk5_architecture-variants.git"
BRANCH = "claude/trusting-pascal-1qjsp0"
COMMIT = ""   # e.g. "d80ca1d..." ; leave empty to use the branch head (the exact hash is printed and recorded either way)

import os, subprocess
if not os.path.isdir("repo"):
    url = REPO_URL
    try:                                   # private repo: add a Colab secret named GITHUB_TOKEN (never printed)
        from google.colab import userdata
        tok = userdata.get("GITHUB_TOKEN")
        if tok:
            url = REPO_URL.replace("https://", f"https://{tok}@")
    except Exception:
        pass
    subprocess.run(["git", "clone", "--branch", BRANCH, url, "repo"], check=True)
    subprocess.run(["git", "-C", "repo", "remote", "set-url", "origin", REPO_URL], check=True)   # drop the token from the remote
if COMMIT:
    subprocess.run(["git", "-C", "repo", "checkout", COMMIT], check=True)
%cd repo/week7_capstone
!git rev-parse HEAD
!git status --short

In [ ]:
# 2. GPU check. The study is defined for a Colab GPU; stop here if there is none.
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU: switch the runtime to a T4 GPU and restart."
print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0))

In [ ]:
# 3. Environment: capture the lock on the first session, verify against it afterwards. Never installs or upgrades.
!python src/check_env.py --auto

In [ ]:
# 4. Determinism policy timing: P1 vs P2 on identical workloads, timing only (no accuracy is computed).
#    It prints the measured difference first, then applies the pre-declared feasibility rule.
!python src/choose_policy.py

In [ ]:
# 5. Smoke checks (V4 determinism, V5 shared initialization, V6 scale, V7 demand floor, V16 memory) and the unit tests.
#    Run after step 4 so no accuracy figure can influence the policy decision.
!python src/smoke_checks.py

In [ ]:
# 6. Summary of what was recorded.
import json
d = json.load(open("results/determinism_decision.json"))
print("selected policy:", d["selected_policy"], "| accuracy used:", d["accuracy_used"], "| feasible:", d["feasible_under_selected"])
print("projected total hours:", d["projected_total_hours"], "| ceiling:", d["ceiling_hours"])
print("measured penalty % (train step, P1 vs P2):", d["measured_penalty_percent_train_step"])
print("GPU:", d["env"]["gpu_name"], "| torch", d["env"]["torch"], "| CUDA", d["env"]["cuda_runtime"], "| commit", d["env"]["git_commit"])
checks = json.load(open("results/smoke/checks/smoke_checks.json"))["checks"]
for c in checks:
    print(f"{c['id']:<58}{c['status']}")
for c in checks:                                   # show the numbers behind anything that did not simply pass
    if c["status"] not in ("PASS", "SKIP"):
        print("\n--- detail:", c["id"], c["status"])
        print(json.dumps(c["detail"], indent=1))

In [ ]:
# 7. Package the smoke outputs (kept separate from any later calibration or main results) and download them.
import glob, subprocess
paths = ["results/smoke", "results/determinism_decision.json", "results/env_report.json", "results/session_manifest.json",
         "requirements.lock.txt"] + glob.glob("results/pip_freeze_*.txt") + glob.glob("data/*.sha256")
paths = [p for p in paths if os.path.exists(p)]
subprocess.run(["zip", "-rq", "week7_smoke_outputs.zip", *paths], check=True)
print("zipped:", paths)
from google.colab import files
files.download("week7_smoke_outputs.zip")

**Stop here.** Send back `week7_smoke_outputs.zip`. Next steps (calibration, pilot, freeze) happen only after the outputs are reviewed. If `requirements.lock.txt` was created in this session, commit it to the repository so later sessions are checked against it.